# Phase 4A — Milestone 3: Cross-Sectional + Regime-Aware Feature Ablation

This notebook walks through and validates **Milestone 3** of Phase 4A: the
per-feature ablation harness shipped in this branch, applied to the **7
pre-committed candidate features**.

**Reference documents**
- PRD: [`phase-4a-feature-and-label-redesign.prd.md`](../.claude/prds/phase-4a-feature-and-label-redesign.prd.md)
- Plan: [`phase-4a-milestone-3-regime-features.plan.md`](../.claude/plans/phase-4a-milestone-3-regime-features.plan.md)
- Concept docs: [`regime-evaluation.md`](../docs/concepts/regime-evaluation.md), [`feature-glossary.md`](../docs/concepts/feature-glossary.md)
- Companion notebooks: [`06_phase4a_label_ablation.ipynb`](06_phase4a_label_ablation.ipynb), [`07_phase4a_fred_leakage.ipynb`](07_phase4a_fred_leakage.ipynb)

**Why this notebook exists.** nb06 (Milestone 2) found that no label scheme
alone fixes the `rate_cycle` failure regime — `signed_returns` stays the
default label and the work hands to *features*. nb07 (Milestone 5) fixed the
FRED publication-lag leak (the corrected lagged join is now the
`build_features` default — every run below uses it) and re-attributed the
IS-macro-dominance puzzle to feature/label instability. Milestone 3 asks two
questions:

1. **Which of the 7 pre-committed candidate features add per-regime OOS edge,
   and which are noise?**
2. **Do SHAP (in-sample importance) and OOS ablation agree on the answer?**

| Candidate | Family | What it adds |
|---|---|---|
| `xs_rank_ret_21d` | cross-sectional | where the symbol's 1-month momentum sits in today's panel |
| `xs_rank_ret_252d` | cross-sectional | 12-month momentum rank — the classic cross-sectional momentum sort |
| `xs_rank_vol_21d` | cross-sectional | realized-vol rank — low-vol-anomaly axis |
| `vix_regime` | regime | ordinal {0,1,2} VIX bucket; thresholds shared with M1's `VIXThresholdDetector` |
| `curve_inverted` | regime | binary `yield_curve < 0` |
| `vol_regime_ratio` | regime | `vol_21d / vol_63d` — near-term vol expansion vs contraction |
| `trend_regime` | regime | binary `ma200_ratio > 1` |

**Pre-committed PRD gate** (pinned in the plan *before* any result was
computed — do not adjust): *"≥ 3 features show ≥ 0.1 Sharpe lift net of costs
in ≥ 1 regime."* `feature_ablation_gate` implements this verbatim, plus a
**noise guard** (on by default): a qualifying lift must either have a paired
21-day block-bootstrap 90% CI on the Sharpe delta that excludes 0, or be
positive in ≥ 2 regime columns (cross-regime sign-consistency). The standard
error of an annualized Sharpe over ~5 years of daily data is several times
the 0.1 threshold, so raw-lift cherry-picking across 7 candidates × several
regimes would mostly select noise (winner's curse).

**A negative finding (no survivors) is a valid outcome** — it gets reported
honestly and Milestone 6 still runs the full-panel confirmation.

**Two-stage discipline.** Everything here runs on the fast 5-symbol ×
~8-year slice (nb06/nb07 convention) with the preview GBM (`n_iter=10`).
The slice verdict is **provisional**: Milestone 6 re-evaluates the surviving
features on the full 33-symbol × 23-year panel, and those numbers supersede
these.

**Sections**
1. Setup and slice panel (corrected FRED joins + 24-column feature matrix)
2. Feature sanity demos (known stretches)
3. Add-one ablation — 8 runs
4. Per-regime lift table + PRD gate
5. SHAP-vs-ablation agreement (Spearman ρ)
6. Leave-one-out spot-check
7. Verdict + hand-off


---
## 1 — Setup and imports


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import sys
from pathlib import Path

ROOT = Path('..').resolve()
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})

print(f'numpy   {np.__version__}')
print(f'pandas  {pd.__version__}')


In [ ]:
# Milestone 3 surface area — everything we will exercise in this notebook.
from quant.features.engineering import build_features
from quant.features.cross_sectional import add_cross_sectional_features
from quant.features.labels import generate_labels
from quant.models.gbm import GBMModel
from quant.backtest.harness import BacktestResult
from quant.backtest.ablation import (
    make_add_one_sets,
    make_leave_one_out_sets,
    run_feature_ablation,
)
from quant.backtest.regimes import DateRangeDetector, tag_regimes
from quant.backtest.report import (
    feature_ablation_gate,
    feature_ablation_table,
    format_feature_ablation_report,
)

import xgboost as xgb

CANDIDATES = [
    'xs_rank_ret_21d',
    'xs_rank_ret_252d',
    'xs_rank_vol_21d',
    'vix_regime',
    'curve_inverted',
    'vol_regime_ratio',
    'trend_regime',
]

print('Milestone 3 surface area imported.')
print(f'Pre-committed candidates ({len(CANDIDATES)}):')
for c in CANDIDATES:
    print(f'  {c}')


### Real-data slice (5 symbols × ~8 years)

Same slice convention as nb06 §4 / nb07 §1 — sized for fast iteration, not
for the final Phase 4A gate. The full-panel measurement belongs to
Milestone 6.


In [ ]:
import quant.storage.catalog as catalog

DEMO_SYMBOLS = ['AAPL', 'MSFT', 'JPM', 'JNJ', 'SPY']
DEMO_START = '2018-01-02'
DEMO_END = '2026-04-21'

REAL_OHLCV_LOADED = False
prices_by_sym: dict[str, pd.DataFrame] = {}
try:
    syms_sql = ', '.join(f"'{s}'" for s in DEMO_SYMBOLS)
    eq = catalog.query(f"""
        SELECT symbol, timestamp, open, high, low, close, adjClose, volume
        FROM {catalog.table('equity_eod_tiingo')}
        WHERE symbol IN ({syms_sql})
          AND timestamp BETWEEN '{DEMO_START}' AND '{DEMO_END}'
        ORDER BY symbol, timestamp
    """)
    eq['timestamp'] = pd.to_datetime(eq['timestamp']).dt.tz_localize(None)
    eq = eq.set_index('timestamp')
    for s in DEMO_SYMBOLS:
        sdf = eq[eq['symbol'] == s][['open', 'high', 'low', 'close', 'volume']].copy()
        sdf = sdf[~sdf.index.duplicated(keep='last')].sort_index()
        if not sdf.empty:
            prices_by_sym[s] = sdf
    REAL_OHLCV_LOADED = bool(prices_by_sym)
    print(f'Loaded OHLCV for {len(prices_by_sym)} symbols')
    for s, df in prices_by_sym.items():
        print(f'  {s}: {len(df):,} bars  {df.index.min().date()} → {df.index.max().date()}')
except Exception as exc:
    print(f'Lake unavailable ({type(exc).__name__}): {exc}')


### Build the 24-column feature panel

`build_features` now produces **21 columns** per symbol (17 base + 4 regime
indicators, Milestone 3) under the **corrected FRED publication-lag joins by
default** (Milestone 5). `add_cross_sectional_features` then appends the 3
`xs_rank_*` columns → **24 columns** total.

Two deliberate choices:

- **`min_symbols=5` on a 5-symbol slice** — every rank pool must contain all
  five symbols, so a rank exists only where every symbol has data. On 5
  symbols the ranks are **coarse quintiles** ({0.2, 0.4, 0.6, 0.8, 1.0});
  the full 33-symbol panel in Milestone 6 gives much finer ranks. The slice
  measures *whether the axis carries signal at all*, not its full resolution.
- **One `dropna()` over all 24 columns** — every ablation run below slices
  its column subset from the *same rows*, so the baseline and every `+c`
  variant see identical fold structure. Row composition is never a
  confounder (the same kwargs-discipline as `run_label_ablation`).

The baseline column list is the **17 base columns by name** — not "whatever
is in the frame" — because the frames now carry 24 columns.


In [ ]:
HORIZON = 1

features_by_sym: dict[str, pd.DataFrame] = {}
labels_by_sym: dict[str, pd.Series] = {}
gbm_prices_by_sym: dict[str, pd.DataFrame] = {}
BASE_17: list[str] = []
ALL_COLS: list[str] = []

if REAL_OHLCV_LOADED:
    feats_raw = build_features(DEMO_SYMBOLS, prices_by_sym)  # corrected FRED joins (M5 default)
    for s in DEMO_SYMBOLS:
        # build_features returns UTC tz-aware indexes (FRED ASOF join);
        # prices are tz-naive. Strip tz so indexes intersect (nb07 §3).
        feats_raw[s].index = feats_raw[s].index.tz_localize(None)

    feats_xs = add_cross_sectional_features(feats_raw, min_symbols=5)

    ALL_COLS = list(feats_xs[DEMO_SYMBOLS[0]].columns)
    BASE_17 = [c for c in ALL_COLS if c not in CANDIDATES]
    assert len(ALL_COLS) == 24, f'expected 24 columns, got {len(ALL_COLS)}: {ALL_COLS}'
    assert len(BASE_17) == 17, f'expected 17 baseline columns, got {len(BASE_17)}: {BASE_17}'
    assert all(c in ALL_COLS for c in CANDIDATES), 'candidate column missing from built frames'

    for s in DEMO_SYMBOLS:
        X = feats_xs[s].dropna()  # one dropna over ALL 24 columns — identical rows for every run
        label = generate_labels(prices_by_sym[s]['close'], horizon=HORIZON)
        y = label.series.dropna()
        common = X.index.intersection(y.index)
        features_by_sym[s] = X.loc[common]
        labels_by_sym[s] = y.loc[common]
        gbm_prices_by_sym[s] = prices_by_sym[s].loc[common]

    n_bars = sum(len(v) for v in labels_by_sym.values())
    print(f'Built panel: {n_bars:,} (sym, bar) pairs × {len(ALL_COLS)} features.')
    print(f'Baseline-17 (by name): {BASE_17}')

    ref_idx = features_by_sym[DEMO_SYMBOLS[0]].index
    shared = all(features_by_sym[s].index.equals(ref_idx) for s in DEMO_SYMBOLS)
    print(f'\nSymbols share one calendar: {shared} '
          f'({len(ref_idx):,} bars/symbol, {ref_idx.min().date()} → {ref_idx.max().date()})')


---
## 2 — Feature sanity demos

Before spending compute on 8 backtests, verify each feature family behaves as
designed on known stretches of history. These are *sanity checks*, not
significance tests.

### 2a — `curve_inverted` flips during 2022–23

The Fed's 2022 hiking cycle inverted the 10y−FFR spread from mid-2022 through
most of 2023–24 — the deepest inversion since 1981. The feature should be ~0
before 2022 and saturate after.


In [ ]:
if REAL_OHLCV_LOADED:
    spy_feat = features_by_sym['SPY']
    inv_by_year = spy_feat['curve_inverted'].groupby(spy_feat.index.year).mean()
    print('Fraction of bars with inverted curve (yield_curve < 0), by year:')
    print(inv_by_year.round(3).to_string())

    fig, ax = plt.subplots(figsize=(7, 2.5))
    inv_by_year.plot.bar(ax=ax, color='black', width=0.7)
    ax.set_ylabel('inverted fraction')
    ax.set_title('curve_inverted occupancy by year (SPY frame)')
    plt.tight_layout()
    plt.show()


### 2b — `xs_rank_ret_21d` distribution is ~uniform

A percentile rank over a fixed 5-symbol pool can only take the values
{0.2, 0.4, 0.6, 0.8, 1.0} (ties aside), each with ~20% mass when pooled
across symbols. Per-symbol mean rank near 0.6 (the mean of those five
values) means no symbol is a permanent leader/laggard at the 1-month horizon
— exactly what a rotating momentum rank should look like.


In [ ]:
if REAL_OHLCV_LOADED:
    pooled = pd.concat([features_by_sym[s]['xs_rank_ret_21d'] for s in DEMO_SYMBOLS])
    print('Pooled distribution of xs_rank_ret_21d (expected ≈ 0.2 mass each):')
    print(pooled.value_counts(normalize=True).sort_index().round(3).to_string())

    per_sym_mean = pd.Series(
        {s: features_by_sym[s]['xs_rank_ret_21d'].mean() for s in DEMO_SYMBOLS}
    )
    print('\nMean rank per symbol (uniform rotation ⇒ ≈ 0.6 for all):')
    print(per_sym_mean.round(3).to_string())


### 2c — `vix_regime` occupancy vs known stretches

`vix_regime` buckets VIXCLS with the same thresholds as M1's
`VIXThresholdDetector` (0 = calm, 1 = normal, 2 = stressed). The COVID crash
(2020) should show a clear regime-2 spike; the calm stretches of 2019 and
2023–24 should be regime-0/1 dominated.


In [ ]:
if REAL_OHLCV_LOADED:
    occ = pd.crosstab(spy_feat.index.year, spy_feat['vix_regime'], normalize='index')
    occ.columns = [f'regime_{int(c)}' for c in occ.columns]
    occ.index.name = 'year'
    print('vix_regime occupancy by year (0=calm, 1=normal, 2=stressed):')
    display(occ.round(3))

    if 2020 in occ.index and 'regime_2' in occ.columns:
        print(f"COVID check — 2020 stressed share: {occ.loc[2020, 'regime_2']:.1%} "
              f"(vs {occ['regime_2'].drop(2020).mean():.1%} mean across other years)")


---
## 3 — Add-one ablation: 8 runs

`make_add_one_sets(BASE_17, CANDIDATES)` builds the baseline set plus one
`+candidate` set per candidate — 8 feature sets. `run_feature_ablation` runs
each through `run_portfolio_backtest` with:

- **GBM preview config** (`n_iter=10`, `n_splits=3`, `random_state=7`) —
  same as nb05 §10 / nb07 §3, and the same seed for every set so the
  hyperparameter candidate list is identical and the *only* difference
  between runs is the feature columns;
- **`signed_returns` labels** (the M2 default), `label_horizon=1`;
- identical `train_window=504 / test_window=63 / step=63 / embargo=3`
  across all sets (kwargs-discipline), with one `deepcopy` of the model per
  set so state cannot leak between runs.


In [ ]:
feature_sets: dict[str, list[str]] = {}
if REAL_OHLCV_LOADED:
    feature_sets = make_add_one_sets(BASE_17, CANDIDATES)
    print(f'{len(feature_sets)} feature sets:')
    for name, cols in feature_sets.items():
        print(f'  {name:<22} {len(cols)} columns')


In [ ]:
GBM_PREVIEW = dict(label_horizon=HORIZON, n_iter=10, n_splits=3, random_state=7)

ablation_results: dict[str, BacktestResult] = {}
if feature_sets:
    ablation_results = run_feature_ablation(
        feature_sets=feature_sets,
        model=GBMModel(**GBM_PREVIEW),
        features_by_symbol=features_by_sym,
        labels_by_symbol=labels_by_sym,
        prices_by_symbol=gbm_prices_by_sym,
        train_window=504,
        test_window=63,
        step=63,
        embargo=3,
        label_horizon=HORIZON,
    )
    print(f'Ran {len(ablation_results)} feature sets through run_feature_ablation.\n')
    for name, res in ablation_results.items():
        m = res.oos_metrics
        print(f'  {name:<22} OOS bars={len(res.oos_returns):5d}  '
              f'Sharpe={m["sharpe"]:+.3f}  MaxDD={m["max_drawdown"]:+.2%}')


---
## 4 — Per-regime lift table + PRD gate

`feature_ablation_table` shows the per-regime Sharpe **delta vs baseline**
(the baseline row is absolute Sharpe — the reference level).
`feature_ablation_gate` then applies the pre-committed PRD metric with the
noise guard.

Regime tagging uses the same `DateRangeDetector` macro-era axis as nb06/nb07.
Note the slice's OOS span starts ~2 years after the panel start
(`train_window=504`), so `qe_bull` is largely or entirely absent here — the
gate evaluates over the regimes the slice actually contains. The full
33-symbol × 23-year panel in Milestone 6 covers all three PRD eras.


In [ ]:
era_labels = None
lift_table = None
if ablation_results:
    base_idx = ablation_results['baseline'].oos_returns.index
    # Identical rows + identical kwargs ⇒ identical fold structure across sets.
    assert all(res.oos_returns.index.equals(base_idx) for res in ablation_results.values())

    era_labels = tag_regimes(base_idx, DateRangeDetector())
    print('Regime composition of the OOS panel:')
    print(era_labels.value_counts().rename('bars').to_frame())

    lift_table = feature_ablation_table(ablation_results, 'baseline', era_labels)
    print('\nPer-regime OOS Sharpe Δ vs baseline (baseline row = absolute Sharpe):')
    display(lift_table.round(3))


In [ ]:
gate = None
survivors: list[str] = []
if ablation_results and era_labels is not None:
    gate = feature_ablation_gate(ablation_results, 'baseline', era_labels)
    print(format_feature_ablation_report(ablation_results, 'baseline', era_labels))

    survivors = [name.removeprefix('+') for name in gate['qualifying_features']]
    print(f"Gate passed: {gate['gate_passed']}  "
          f"({len(gate['qualifying_features'])}/{gate['thresholds']['min_features']} qualifying, "
          f"noise guard on)")
    print(f"Survivors: {survivors if survivors else '(none)'}")


---
## 5 — SHAP-vs-ablation agreement

The PRD asks that *"SHAP and OOS performance agree on dominant features."*
This section quantifies that as a **Spearman rank correlation** between:

- each candidate's **IS mean-|SHAP| importance** from one GBM trained on the
  combined 24-column set (nb03's pattern, via the booster's native
  `pred_contribs` — no `shap` library needed; preview `n_iter=10` to keep it
  cheap), and
- each candidate's **best-regime OOS ablation lift** from §4's table.

**IS-only fit — interpretation only** (same caveat as nb03/nb07 §4b): the
GBM is trained on the entire stacked slice with no walk-forward, so absolute
SHAP values are inflated; only the *ordering* is used. The ρ is **reported,
not gated** — a low ρ is itself a finding (IS importance does not transfer
OOS, the nb03 puzzle again).


In [ ]:
shap_imp = None
if REAL_OHLCV_LOADED:
    def stack_panel(cols: list[str]):
        X = np.vstack([features_by_sym[s][cols].to_numpy() for s in DEMO_SYMBOLS])
        y = np.concatenate([labels_by_sym[s].to_numpy() for s in DEMO_SYMBOLS])
        return X, y

    X_all, y_all = stack_panel(ALL_COLS)
    gbm_is = GBMModel(label_horizon=HORIZON, n_iter=10, n_splits=3, random_state=42)
    gbm_is.fit(X_all, y_all)

    booster = gbm_is._model.get_booster()
    dmat = xgb.DMatrix(X_all, feature_names=ALL_COLS)
    contribs = booster.predict(dmat, pred_contribs=True)[:, :-1]
    shap_imp = pd.Series(
        np.abs(contribs).mean(axis=0), index=ALL_COLS
    ).sort_values(ascending=False)

    print(f'IS GBM trained on {X_all.shape[0]:,} stacked rows × {X_all.shape[1]} features.')
    print('\nIS mean-|SHAP| importance — top 10 of 24:')
    print(shap_imp.head(10).round(5).to_string())

    full_rank = shap_imp.rank(ascending=False)
    print('\nCandidate positions in the full 24-feature SHAP ordering:')
    for c in CANDIDATES:
        print(f'  {c:<20} mean|SHAP|={shap_imp[c]:.5f}  rank {int(full_rank[c]):>2}/24')


In [ ]:
spearman_rho = None
best_lift = None
if shap_imp is not None and lift_table is not None:
    regime_cols = [c for c in lift_table.columns if c not in ('aggregate', 'n_bars')]
    # Best-regime lift per candidate — the same quantity the gate qualifies on.
    best_lift = pd.Series(
        {c: lift_table.loc[f'+{c}', regime_cols].max() for c in CANDIDATES},
        name='best_regime_lift',
    )
    cand_shap = shap_imp[CANDIDATES].rename('mean_abs_shap')

    agreement = pd.concat([cand_shap, best_lift], axis=1)
    agreement['shap_rank'] = agreement['mean_abs_shap'].rank(ascending=False)
    agreement['lift_rank'] = agreement['best_regime_lift'].rank(ascending=False)
    display(agreement.round(4))

    spearman_rho = float(
        agreement['shap_rank'].corr(agreement['lift_rank'], method='spearman')
    )
    print(f'Spearman ρ (IS SHAP rank vs best-regime OOS lift rank, '
          f'{len(CANDIDATES)} candidates): {spearman_rho:+.3f}')
    print('Reported, not gated — quantifies the PRD\'s "SHAP and OOS agree" ask.')


---
## 6 — Leave-one-out spot-check

The add-one design measures each candidate *in isolation* on top of the
baseline. The complementary question: with the chosen candidates *all
present*, does removing any single one hurt? If yes (negative Δ on its `-c`
row), its add-one lift was not just slack picked up by a correlated sibling.

Set selection: the LOO runs on **(17 + survivors)** if §4 produced
survivors; otherwise on **(17 + top-2 by best-regime lift)** as a robustness
illustration. `make_leave_one_out_sets` builds the full sweep; the slice
pass keeps only `all` + the **candidate** removals — ablating the 17 base
columns is out of scope for M3 (they were validated in Phase 2/2.5).


In [ ]:
loo_results: dict[str, BacktestResult] = {}
loo_table = None
chosen: list[str] = []
if gate is not None and best_lift is not None:
    if survivors:
        chosen = list(survivors)
        print(f'Survivors exist — LOO over the survivor set: {chosen}')
    else:
        chosen = list(best_lift.sort_values(ascending=False).head(2).index)
        print(f'No survivors — LOO over top-2 by best-regime lift '
              f'(robustness illustration): {chosen}')

    combined = BASE_17 + chosen
    loo_all = make_leave_one_out_sets(combined)
    loo_sets = {
        k: v for k, v in loo_all.items()
        if k == 'all' or k.removeprefix('-') in chosen
    }
    print(f'LOO sets ({len(loo_sets)} runs): {list(loo_sets)}')

    loo_results = run_feature_ablation(
        feature_sets=loo_sets,
        model=GBMModel(**GBM_PREVIEW),
        features_by_symbol=features_by_sym,
        labels_by_symbol=labels_by_sym,
        prices_by_symbol=gbm_prices_by_sym,
        train_window=504,
        test_window=63,
        step=63,
        embargo=3,
        label_horizon=HORIZON,
    )
    era_loo = tag_regimes(loo_results['all'].oos_returns.index, DateRangeDetector())
    loo_table = feature_ablation_table(loo_results, 'all', era_loo)
    print("\nPer-regime OOS Sharpe Δ vs 'all' "
          "(negative Δ on a '-c' row ⇒ removing c hurt ⇒ c contributes):")
    display(loo_table.round(3))


---
## 7 — Verdict + hand-off

The cell below renders the slice-level verdict from the measured numbers.
Reminder of the two-stage discipline: this verdict is **provisional** —
Milestone 6 re-runs the survivors on the full 33-symbol × 23-year panel
(all three PRD eras, finer cross-sectional ranks) and those numbers
supersede these.

**Hand-off:**
- **Survivors** → registered in the Milestone 4 feature catalog and included
  in the Milestone 6 full-panel run.
- **Non-survivors** → documented as noise *on this slice* in the feature
  glossary; they remain in `build_features` output (they are cheap and
  point-in-time correct) but carry no validated-edge claim.
- **New feature ideas surfaced mid-milestone** go to the glossary's "Future
  candidates (parking lot)" subsection — not into this ablation. The
  candidate list was pre-committed; widening it after seeing results would
  reintroduce the winner's curse the noise guard exists to prevent.


In [ ]:
if gate is not None:
    noise = [c for c in CANDIDATES if c not in survivors]
    print('=' * 70)
    print('MILESTONE 3 SLICE VERDICT (provisional — M6 confirms at full panel)')
    print('=' * 70)
    print(f"PRD gate (≥3 features, ≥0.1 Sharpe lift, ≥1 regime, noise guard on): "
          f"{'PASSED' if gate['gate_passed'] else 'FAILED'}")
    print(f"Qualifying features: {len(gate['qualifying_features'])}/{len(CANDIDATES)} candidates")
    print(f"Survivors → M4 registration + M6 full-panel run: "
          f"{survivors if survivors else '(none)'}")
    print(f"Documented as noise on this slice: {noise}")
    if spearman_rho is not None:
        print(f"SHAP-vs-ablation agreement (Spearman ρ): {spearman_rho:+.3f}")
    if chosen and loo_table is not None:
        print(f"LOO spot-check set: {chosen}")


### Measured verdict (placeholder)

*This cell is filled in with the measured numbers after execution.*
